# Lab 3 · Part 1 — From notebook to a model others can use
### EUROCONTROL AI Lab · Exercise 3: an end-to-end ML pipeline

In Labs 1 and 2 the model lived only inside a notebook. In operations, a model is useful only if **another system can ask it for predictions**.
In this lab we build the complete path, entirely on your own computer — no cloud service:

```
 METAR archive → prepare data → train → test → save the model → local prediction service → client / web form
               └──────────────── Part 1 (this notebook) ─────────┘ └─────────── Part 2 ─────────────────┘
```

We reuse the problem of Lab 2: the probability that visibility at Malpensa drops **below 1 500 m within 3 hours** — our **proxy** for Low-Visibility Procedures.

| | |
|---|---|
| ⏱ **Time** | Part 1 about 40 min · Part 2 about 1 hour |

**What you will learn**
1. Why the data preparation is put in **one shared file** used by both training and the service.
2. How the training data defines the **input "contract"** of a service (fields, units, valid ranges).
3. How to **save** a model with a **model card**: everything needed to reuse it correctly — and an honest description of what it predicts.
4. How to run the model as a **local web service** and test it (Part 2).

> **How to use this notebook**
> - Run the cells **one at a time, from top to bottom**: click a cell and press **Shift + Enter**.
> - Every line of code has a comment (the text after `#`) explaining what it does. You do not need to write any code.
> - After each plot there is a short **📖 Reading** that explains what the plot tells us.
> - If something breaks, restart and run everything again: *Kernel → Restart Kernel and Run All Cells* (JupyterLab) or *Runtime → Restart session and run all* (Colab).
> - This notebook is yours to keep: you can re-run it later and change the numbers to experiment.

> ☁️ **Running in Google Colab?** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dblue-tech/ai-lab/blob/main/lab3_serving/Lab3_part1_train_and_save.ipynb)  
> Run the next cell **first**: it copies the lab files (data, code, trained models) from GitHub into Colab, so that the rest of the notebook works exactly as on a PC. Wait for *"Colab ready"* (about half a minute).  
> To keep your own copy with your results: *File → Save a copy in Drive*.  
> **On a lab PC** the next cell does nothing: just run it and continue.

In [1]:
# ---- Only for Google Colab: get the lab files. On a lab PC this cell does nothing. ----
import os                                          # os: folders and files
import sys                                         # sys: information about the running Python
import subprocess                                  # subprocess: runs other programs (git, pip)

IN_COLAB = "google.colab" in sys.modules           # True when this notebook runs in Google Colab
if IN_COLAB:                                       # in Colab only:
    if not os.path.exists("/content/ai-lab"):      # if the lab files are not here yet...
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dblue-tech/ai-lab.git", "/content/ai-lab"], check=True)   # ...copy them from GitHub
    os.chdir("/content/ai-lab/lab3_serving")       # work inside this notebook's folder, as on a PC
    print("Colab ready — working in", os.getcwd())  # confirm
else:                                              # on a lab PC:
    print("Running on this computer: nothing to do")   # the files are already here

Running on this computer: nothing to do


---
## 0 · Setup
The data preparation and the model are now in the file **`lvp_core.py`** (in this folder — you can open it in JupyterLab; every line is commented).
This notebook only *uses* it.

In [2]:
import sys                                        # sys: access to Python's list of code folders
import json                                       # json: reads the model card
import time                                       # time: measures durations
from pathlib import Path                          # Path: file paths that work on every system
import numpy as np                                # numpy: calculations on arrays
import pandas as pd                               # pandas: tables of data
import matplotlib.pyplot as plt                   # matplotlib: plots
import seaborn as sns                             # seaborn: heatmaps
import torch                                      # PyTorch: neural networks

LAB_DIR = Path(".").resolve()                     # this lab's folder (where lvp_core.py and serve.py are)
sys.path.insert(0, str(LAB_DIR))                  # let Python find lvp_core.py
DATA_DIR = LAB_DIR.parent / "data"                # the data folder, next to this lab's folder
ARTIFACTS = LAB_DIR / "artifacts"                 # where the trained model is saved
SEED = 42                                         # fixed seed for random numbers...
torch.manual_seed(SEED)                           # ...in PyTorch...
np.random.seed(SEED)                              # ...and in numpy

from sklearn.metrics import average_precision_score, precision_score, recall_score   # classification measures
import lvp_core                                   # our shared file: data preparation + model
print("Model inputs:", lvp_core.FEATURES)         # the 11 inputs of the model

# ---- Plot style used in the whole notebook (you can skip reading this part) ----
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]  # 8 colours readable by colour-blind people
plt.rcParams["figure.figsize"] = (11, 4)                    # default figure size: 11 x 4 inches
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=PALETTE)  # use our colours, in this order, for the lines
plt.rcParams["axes.grid"] = True                            # draw a grid behind every plot
plt.rcParams["grid.color"] = "#e6e5e0"                      # make the grid light grey
plt.rcParams["axes.spines.top"] = False                     # hide the top border of the plot
plt.rcParams["axes.spines.right"] = False                   # hide the right border of the plot
plt.rcParams["axes.titleweight"] = "bold"                   # bold titles
plt.rcParams["lines.linewidth"] = 2                         # slightly thicker lines
plt.rcParams["legend.frameon"] = False                      # no box around the legend

Model inputs: ['temp_c', 'spread_c', 'rh', 'wind_east', 'wind_north', 'qnh_hpa', 'vis_m', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']


---
## 1 · The data and the input contract
The service will receive observations in **aviation units** (°C, knots, metres, hPa). `load_iem_metar` reads the archive and converts it to exactly those units.

In [3]:
weather = lvp_core.load_iem_metar(DATA_DIR / "LIMC.csv")   # read the archive, one row per hour, aviation units
print("Rows, columns:", weather.shape)                     # size of the table
weather.head()                                             # first 5 rows

Rows, columns: (52584, 7)


,time,temperature_c,dewpoint_c,wind_dir_deg,wind_speed_kt,visibility_m,qnh_hpa
valid,,,,,,,
2020-01-01 00:50:00,2020-01-01 00:50:00,-2.0,-2.0,360.0,6.0,9994.0014,1029.123921
2020-01-01 01:50:00,2020-01-01 01:50:00,-2.0,-2.0,10.0,7.0,9994.0014,1029.123921
2020-01-01 02:50:00,2020-01-01 02:50:00,-2.0,-2.0,20.0,5.0,9994.0014,1029.123921
2020-01-01 03:50:00,2020-01-01 03:50:00,-3.0,-3.0,360.0,5.0,9994.0014,1029.123921
2020-01-01 04:50:00,2020-01-01 04:50:00,-2.0,-2.0,360.0,5.0,9994.0014,1029.123921


A service must **refuse inputs that make no sense** (a temperature of 70 °C is a unit error, not a heat wave).
We look at the range of each input in the training years: these ranges become the limits the service accepts.

In [4]:
training_years = weather[weather.index < "2024-01-01"]     # the training period (2020–2023)
inputs_only = training_years.drop(columns="time")          # all columns except the time
ranges = pd.DataFrame()                                    # an empty table
ranges["minimum"] = inputs_only.min()                      # smallest value of each input
ranges["maximum"] = inputs_only.max()                      # largest value of each input
ranges["% missing"] = inputs_only.isna().mean() * 100      # share of missing values
ranges.round(1)                                            # display, rounded

,minimum,maximum,% missing
temperature_c,-7.0,37.0,0.6
dewpoint_c,-21.0,24.0,0.6
wind_dir_deg,0.0,360.0,19.9
wind_speed_kt,0.0,31.0,0.6
visibility_m,0.0,9994.0,0.6
qnh_hpa,982.1,1044.0,0.6


📖 **Reading → the limits used by the service** (see the class `Observation` in `serve.py`):
- temperature −30…50 °C, dew point −40…35 °C and never above the temperature · wind 0…80 kt · QNH 950…1 060 hPa
- visibility 0…9 999 m — **9 999 means "10 km or more"**
- wind direction may be **empty** = calm or variable wind

The limits are a bit wider than what was observed (a safety margin), but they block obvious unit errors.

---
## 2 · Build the training data with the shared functions

In [5]:
X_table = lvp_core.build_features(weather)                 # operational inputs -> the 11 model inputs
y_series = lvp_core.build_target(weather)                  # the proxy target: visibility < 1500 m within 3 hours
data = X_table.copy()                                      # copy the inputs...
data["target"] = y_series                                  # ...add the target...
data = data.dropna()                                       # ...and remove incomplete hours

is_train = data.index < "2024-01-01"                       # 2020–2023
is_validation = (data.index >= "2024-01-01") & (data.index < "2025-01-01")   # 2024
is_test = data.index >= "2025-01-01"                       # 2025

X = data[lvp_core.FEATURES].values.astype("float32")       # inputs as a table of numbers
y = data["target"].values.astype("float32")                # targets as 0/1
means = X[is_train].mean(axis=0)                           # scaling averages (training only)
spreads = X[is_train].std(axis=0)                          # scaling spreads (training only)
X_scaled = (X - means) / spreads                           # scaled inputs

print("Training:", is_train.sum(), "hours · validation:", is_validation.sum(), "· test:", is_test.sum())   # sizes

Training: 34392 hours · validation: 8643 · test: 8564


---
## 3 · Train with the best settings from Lab 2
We do not repeat the search for good settings: we reuse the best ones from Lab 2 and write them in the model card.

> 🔍 **Want to see the training loop at work?** Exercise 2, §6.0 (or 2S-1, §7.0) prints a few of its steps, only for studying:
> for the first batch of some epochs, the inputs the model receives, its answers and the truth.
> This notebook has no such demonstration on purpose: the model trained below is **the one that is saved and served in Part 2**, and nothing else should touch it.

In [6]:
SETTINGS = {"hidden": 32, "dropout": 0.1, "learning_rate": 0.001, "epochs": 20, "batch_size": 256, "pos_weight": 3.0}   # from Lab 2

torch.manual_seed(SEED)                                                          # fixed random start
model = lvp_core.FogMLP(len(lvp_core.FEATURES), SETTINGS["hidden"], SETTINGS["dropout"])   # a new model
loss_function = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(SETTINGS["pos_weight"]))  # loss with class weight
optimizer = torch.optim.Adam(model.parameters(), lr=SETTINGS["learning_rate"])   # optimizer
train_data = torch.utils.data.TensorDataset(torch.tensor(X_scaled[is_train]), torch.tensor(y[is_train]))   # training examples
loader = torch.utils.data.DataLoader(train_data, batch_size=SETTINGS["batch_size"], shuffle=True)          # batches

start = time.time()                                        # remember the start time
for epoch in range(SETTINGS["epochs"]):                    # repeat for each epoch
    model.train()                                          # training mode
    for inputs, targets in loader:                         # for each batch:
        optimizer.zero_grad()                              #   1. reset corrections
        loss = loss_function(model(inputs), targets)       #   2+3. forecast and measure the error
        loss.backward()                                    #   4. compute corrections
        optimizer.step()                                   #   5. apply corrections
print("Trained in", round(time.time() - start, 1), "seconds")   # training time


def probabilities_for(rows):                               # probabilities of the model for the selected rows
    model.eval()                                           # evaluation mode (dropout off)
    with torch.no_grad():                                  # no corrections
        scores = model(torch.tensor(X_scaled[rows]))       # raw scores
    return torch.sigmoid(scores).numpy()                   # scores -> probabilities


p_validation = probabilities_for(is_validation)            # probabilities on 2024
print("Validation PR-AUC:", round(average_precision_score(y[is_validation], p_validation), 3))   # quality on 2024

Trained in 6.0 seconds
Validation PR-AUC: 0.69


### 3.1 Alert threshold (chosen on 2024) and final test (on 2025)

In [7]:
THRESHOLD = 0.01                                           # start from the lowest threshold...
for candidate in np.arange(0.01, 1.0, 0.01):               # ...and try 0.01, 0.02, ..., 0.99:
    alerts = (p_validation >= candidate).astype(float)     # alerts with this threshold
    if recall_score(y[is_validation], alerts) >= 0.80:     # if it still catches at least 80 % of the cases...
        THRESHOLD = float(candidate)                       # ...remember it (we keep the highest such threshold)
print("Alert threshold:", round(THRESHOLD, 2))             # show the chosen threshold

p_test = probabilities_for(is_test)                        # probabilities on 2025
test_alerts = (p_test >= THRESHOLD).astype(float)          # alerts on 2025
metrics = {}                                               # the quality measures, for the model card
metrics["validation_pr_auc"] = float(average_precision_score(y[is_validation], p_validation))   # PR-AUC on 2024
metrics["test_pr_auc"] = float(average_precision_score(y[is_test], p_test))                     # PR-AUC on 2025
metrics["test_precision"] = float(precision_score(y[is_test], test_alerts))                     # precision on 2025
metrics["test_recall"] = float(recall_score(y[is_test], test_alerts))                           # recall on 2025
for name in metrics:                                       # print each measure
    print(name, "=", round(metrics[name], 3))              # name and value

Alert threshold: 0.23
validation_pr_auc = 0.69
test_pr_auc = 0.523
test_precision = 0.331
test_recall = 0.775


---
## 4 · Save the model — weights **and** model card

The learned weights alone are **not enough** to use the model. The service also needs the list and order of the inputs, the scaling values and the alert threshold.
And people need to know what the model is, what data it saw, how good it is — and **what the target really means**: ours is a *proxy* for LVP, and the card must say so,
otherwise a future user might believe the model predicts actual LVP declarations. This description is called a **model card**.

In [8]:
card = {}                                                  # the model card: a dictionary of labelled information
card["name"] = "Low-visibility nowcast LIMC (LVP proxy)"   # name
card["version"] = "1.0.0"                                  # version number
card["created_utc"] = pd.Timestamp.now("UTC").isoformat(timespec="seconds")   # creation time
card["task"] = "P(visibility < 1500 m in any of the next 3 hourly METARs)"    # what it predicts
card["target_note"] = ("Proxy for Low-Visibility Procedures: derived from METAR visibility only. Real LVP depend on "
                       "RVR, cloud ceiling and local procedures; no LVP declaration data was used.")   # honest description
card["airport"] = "LIMC"                                   # airport
card["training_data"] = {"source": "Iowa Environmental Mesonet METAR archive",
                         "train": "2020-2023", "validation": "2024", "test": "2025"}   # data used
card["features"] = lvp_core.FEATURES                       # the inputs, in order
card["scaler_mean"] = means.tolist()                       # scaling averages (as a plain list)
card["scaler_std"] = spreads.tolist()                      # scaling spreads (as a plain list)
card["threshold"] = THRESHOLD                              # alert threshold
card["threshold_rule"] = "highest threshold with recall >= 0.80 on validation"   # how it was chosen
card["hyperparameters"] = SETTINGS                         # training settings
card["metrics"] = metrics                                  # quality measures
card["intended_use"] = "Training exercise only. Not for operational use."        # allowed use

lvp_core.save_artifacts(model, card, ARTIFACTS)            # save the weights and the card in the artifacts folder
for saved_file in ARTIFACTS.iterdir():                    # for each file in the artifacts folder...
    print("Saved:", saved_file.name)                       # ...show its name

Saved: model_card.json
Saved: model_state.pt


In [9]:
print((ARTIFACTS / "model_card.json").read_text())         # show the saved model card

{
  "name": "Low-visibility nowcast LIMC (LVP proxy)",
  "version": "1.0.0",
  "created_utc": "2026-10-07T21:09:48+00:00",
  "task": "P(visibility < 1500 m in any of the next 3 hourly METARs)",
  "target_note": "Proxy for Low-Visibility Procedures: derived from METAR visibility only. Real LVP depend on RVR, cloud ceiling and local procedures; no LVP declaration data was used.",
  "airport": "LIMC",
  "training_data": {
    "source": "Iowa Environmental Mesonet METAR archive",
    "train": "2020-2023",
    "validation": "2024",
    "test": "2025"
  },
  "features": [
    "temp_c",
    "spread_c",
    "rh",
    "wind_east",
    "wind_north",
    "qnh_hpa",
    "vis_m",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos"
  ],
  "scaler_mean": [
    13.985548973083496,
    6.191730499267578,
    70.98208618164062,
    0.3082016110420227,
    1.8217982053756714,
    1016.2053833007812,
    9032.6572265625,
    0.0025875475257635117,
    -0.005853959359228611,
    -0.00464640604

### 4.1 Always test what you saved
We load the model **from disk**, exactly as the service will, starting again from the **raw operational inputs**, and check that it gives the same answers as the model in memory.

In [10]:
reloaded = lvp_core.LoadedModel(ARTIFACTS)                 # load weights + model card from disk
test_rows = weather.loc[data.index[is_test]]               # the raw operational inputs of the 2025 test hours
p_reloaded = reloaded.predict_probability(test_rows)       # probabilities computed by the reloaded model

difference = np.abs(p_reloaded - p_test)                   # difference, hour by hour
print("Largest difference:", difference.max())             # should be practically zero
if difference.max() < 0.00001:                             # if the difference is negligible...
    print("✅ The saved model gives the same answers as the notebook")   # ...success
else:                                                      # otherwise...
    print("❌ Something is different between training and the saved model")   # ...a problem to investigate

Largest difference: 0.0
✅ The saved model gives the same answers as the notebook


**➡️ Next:** open **Part 2** to run the model as a service.